# DiaSight — 00 Prototype baseline (EfficientNet-B0 @ 224)

Goal: get a **real, small trained model** into the app fast, so the prototype demo shows real predictions.
This is the "baseline" row of the build plan, not the final model (that is EfficientNet-B3 @ 384 in `03_train`).

**Before running:** Runtime → Change runtime type → **T4 GPU**. Put your `kaggle.json` in `MyDrive/DiaSight/` and accept the
APTOS 2019 competition rules on Kaggle. Total time is roughly 40–60 minutes, most of it download + preprocessing.

**Output (in `MyDrive/DiaSight/`):** `models/best.pt`, `models/config.json`, `samples/` (one image per grade), `figures/` (metric tables, confusion matrices, APTOS-vs-IDRiD comparison).

Section 8 (IDRiD) can be run later; the app only needs sections 1-7.
Copy those into the repo's `models/` and `app/samples/` folders and the app switches out of demo mode.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/DiaSight'
for d in ['models', 'samples', 'figures', 'splits']:
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)

In [ ]:
# Get the shared diasight package (same code the app uses)
REPO_URL = 'https://github.com/YOUR_USERNAME/diasight.git'   # <-- change this
!git clone -q $REPO_URL /content/diasight 2>/dev/null || git -C /content/diasight pull -q
%cd /content/diasight
!pip install -q timm grad-cam albumentations kaggle
import sys; sys.path.insert(0, '/content/diasight')

## 1. Download APTOS 2019 (to local Colab disk, not Drive — the zip is ~10 GB)

In [ ]:
!mkdir -p ~/.kaggle && cp "$DRIVE/kaggle.json" ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
if not os.path.exists('/content/aptos/train.csv'):
    !kaggle competitions download -c aptos2019-blindness-detection -p /content
    !unzip -q -o /content/aptos2019-blindness-detection.zip "train_images/*" "train.csv" -d /content/aptos
    !rm /content/aptos2019-blindness-detection.zip
import pandas as pd
df = pd.read_csv('/content/aptos/train.csv')
print(len(df), 'labelled images')
print(df.diagnosis.value_counts().sort_index())

## 2. Preprocess once and cache (crop → resize → Ben Graham → circle), same code as the app

In [ ]:
import cv2, numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from diasight.preprocess import preprocess

SIZE, SIGMA = 224, 10
CACHE = f'/content/cache{SIZE}'
os.makedirs(CACHE, exist_ok=True)

def work(idc):
    out = f'{CACHE}/{idc}.png'
    if not os.path.exists(out):
        rgb = cv2.cvtColor(cv2.imread(f'/content/aptos/train_images/{idc}.png'), cv2.COLOR_BGR2RGB)
        _, model_img = preprocess(rgb, SIZE, SIGMA)
        cv2.imwrite(out, cv2.cvtColor(model_img, cv2.COLOR_RGB2BGR))
    return out

with ThreadPoolExecutor(8) as ex:
    df['path'] = list(tqdm(ex.map(work, df.id_code), total=len(df)))
!cd /content && zip -q -r "$DRIVE/cache{SIZE}.zip" cache{SIZE}   # backup so you never redo this

## 3. Calibrate the quality-check thresholds on real dataset images

In [ ]:
from diasight.quality import calibrate
sample = df.sample(300, random_state=42)
imgs = [cv2.cvtColor(cv2.imread(f'/content/aptos/train_images/{i}.png'), cv2.COLOR_BGR2RGB) for i in tqdm(sample.id_code)]
QUALITY = calibrate(imgs)
print(QUALITY)
del imgs

## 4. Stratified 80/20 split, seed 42

In [ ]:
from sklearn.model_selection import train_test_split
tr, va = train_test_split(df, test_size=0.2, stratify=df.diagnosis, random_state=42)
tr.to_csv(f'{DRIVE}/splits/train.csv', index=False); va.to_csv(f'{DRIVE}/splits/val.csv', index=False)
print(len(tr), 'train /', len(va), 'val')

## 5. Train EfficientNet-B0, classification head, class weights

In [ ]:
import torch, time
from torch.utils.data import DataLoader
from sklearn.metrics import cohen_kappa_score
from diasight.config import DEFAULT_CONFIG
from diasight.dataset import RetinaDataset, train_augs
from diasight.model import build_model

torch.manual_seed(42); np.random.seed(42)
cfg = {**DEFAULT_CONFIG, 'model_name': 'efficientnet_b0', 'img_size': SIZE, 'ben_graham_sigma': SIGMA,
       'head': 'classification', 'quality': QUALITY}
EPOCHS, BS, LR = 8, 32, 1e-4

train_dl = DataLoader(RetinaDataset(tr, cfg, train_augs(SIZE)), BS, shuffle=True, num_workers=2, drop_last=True)
val_dl = DataLoader(RetinaDataset(va, cfg), 64, num_workers=2)

model = build_model(cfg['model_name'], 'classification', pretrained=True).cuda()
counts = np.bincount(tr.diagnosis, minlength=5)
weights = torch.tensor(len(tr) / (5 * counts), dtype=torch.float32).cuda()   # inverse frequency
crit = torch.nn.CrossEntropyLoss(weight=weights)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
total, warm = EPOCHS * len(train_dl), len(train_dl)                          # 1 warm-up epoch, then cosine
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / (total - warm))))
scaler = torch.amp.GradScaler('cuda')

def evaluate():
    model.eval(); P, Y = [], []
    with torch.no_grad(), torch.autocast('cuda'):
        for x, y in val_dl:
            P.append(torch.softmax(model(x.cuda()), 1).float().cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

best_k, history = -1, []
for ep in range(EPOCHS):
    model.train(); t0, run = time.time(), 0.0
    for x, y in train_dl:
        x, y = x.cuda(), y.cuda(); opt.zero_grad()
        with torch.autocast('cuda'):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
    P, Y = evaluate()
    k = cohen_kappa_score(Y, P.argmax(1), weights='quadratic')
    ref_true, ref_pred = Y >= 2, P[:, 2:].sum(1) >= 0.5
    sens = (ref_pred & ref_true).sum() / ref_true.sum()
    history.append((ep + 1, run / len(train_dl), k, sens))
    print(f'epoch {ep+1}: loss {run/len(train_dl):.3f}  val kappa {k:.3f}  refer sensitivity {sens:.3f}  ({time.time()-t0:.0f}s)')
    if k > best_k:
        best_k, best_P = k, P
        torch.save({'model': model.state_dict()}, f'{DRIVE}/models/best.pt')
print('best val kappa', round(best_k, 3))

## 6. Evaluate on APTOS validation (instructor's metric list + refer metrics, with 95% CIs)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from diasight import metrics
P, Y = best_P, va.diagnosis.values
pred, refer_score = P.argmax(1), P[:, 2:].sum(1)
val_summary, val_per_class, val_cm = metrics.report(Y, pred, refer_score, 0.5, 'APTOS validation')

print('Refer cut-off   sensitivity   specificity   (pick the cut-off for high sensitivity in Week 6)')
for th in [0.2, 0.3, 0.4, 0.5]:
    s, sp = metrics._refer(Y, refer_score, th); print(f'    {th:.1f}           {s:.3f}         {sp:.3f}')

ConfusionMatrixDisplay(val_cm.values, display_labels=range(5)).plot(colorbar=False)
plt.title('APTOS validation: 5-grade confusion'); plt.savefig(f'{DRIVE}/figures/val_confusion.png', dpi=150); plt.show()
for name, d in [('val_summary', val_summary), ('val_per_class', val_per_class)]:
    d.to_csv(f'{DRIVE}/figures/{name}.csv')

## 7. Save config.json and one sample image per grade for the app

In [ ]:
import json
from PIL import Image
REFER_THRESHOLD = 0.5   # keep 0.5 for the prototype; tune for high sensitivity in Week 6
cfg['refer_threshold'] = REFER_THRESHOLD
cfg['trained_on'] = 'APTOS 2019 train split (80%), seed 42'
cfg['val_kappa'] = float(best_k)
json.dump(cfg, open(f'{DRIVE}/models/config.json', 'w'), indent=2)

# samples: from the VALIDATION set (never trained on), one per grade, correctly predicted if possible
va2 = va.assign(pred=pred)
for g in range(5):
    pool = va2[(va2.diagnosis == g) & (va2.pred == g)]
    row = (pool if len(pool) else va2[va2.diagnosis == g]).sample(1, random_state=42).iloc[0]
    im = Image.open(f'/content/aptos/train_images/{row.id_code}.png').convert('RGB')
    im.thumbnail((1024, 1024))
    im.save(f'{DRIVE}/samples/grade{g}_{row.id_code}.jpg', quality=92)
print(os.listdir(f'{DRIVE}/samples'))
!cd "$DRIVE" && zip -q -r /content/diasight_prototype_model.zip models samples figures
from google.colab import files; files.download('/content/diasight_prototype_model.zip')

## 8. External test on IDRiD (never trained on)

Download **B. Disease Grading** from IEEE DataPort (free login), upload the zip to `MyDrive/DiaSight/` and set the name below.
IDRiD uses the same international 0-4 scale as APTOS, so labels are compatible. All 516 images (its train + test
splits) are used only as a test set. A drop from APTOS to IDRiD is a domain-shift finding: report it honestly.

In [ ]:
import glob
IDRID_ZIP = f'{DRIVE}/B. Disease Grading.zip'          # <-- change if your file is named differently
!unzip -q -o "$IDRID_ZIP" -d /content/idrid
csvs = [c for c in glob.glob('/content/idrid/**/*.csv', recursive=True) if 'Grading' in c]
lab = pd.concat([pd.read_csv(c) for c in csvs]).dropna(subset=['Image name'])
lab = lab.rename(columns=lambda c: c.strip())
imgs = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob('/content/idrid/**/*.jpg', recursive=True)}
lab['src'] = lab['Image name'].str.strip().map(imgs)
lab = lab.dropna(subset=['src']); print(len(lab), 'IDRiD images with labels')
print(lab['Retinopathy grade'].value_counts().sort_index())

ID_CACHE = f'/content/idrid_cache{SIZE}'; os.makedirs(ID_CACHE, exist_ok=True)
def work_id(row):
    out = f"{ID_CACHE}/{row['Image name'].strip()}.png"
    if not os.path.exists(out):
        rgb = cv2.cvtColor(cv2.imread(row['src']), cv2.COLOR_BGR2RGB)
        cv2.imwrite(out, cv2.cvtColor(preprocess(rgb, SIZE, SIGMA)[1], cv2.COLOR_RGB2BGR))
    return out
lab['path'] = [work_id(r) for _, r in tqdm(lab.iterrows(), total=len(lab))]
lab['diagnosis'] = lab['Retinopathy grade'].astype(int)

model.load_state_dict(torch.load(f'{DRIVE}/models/best.pt')['model'])
model.eval(); PI = []
with torch.no_grad(), torch.autocast('cuda'):
    for x, _ in DataLoader(RetinaDataset(lab, cfg), 32, num_workers=2):
        PI.append(torch.softmax(model(x.cuda()), 1).float().cpu())
PI = torch.cat(PI).numpy()
id_summary, id_per_class, id_cm = metrics.report(lab.diagnosis.values, PI.argmax(1), PI[:, 2:].sum(1),
                                                 cfg['refer_threshold'], 'IDRiD external test')
side = pd.DataFrame({'APTOS val': val_summary['value'], 'IDRiD': id_summary['value']})
print(side); side.to_csv(f'{DRIVE}/figures/aptos_vs_idrid.csv')
id_per_class.to_csv(f'{DRIVE}/figures/idrid_per_class.csv')

**Next:** unzip into the repo — `models/best.pt`, `models/config.json` → `models/`; `samples/*` → `app/samples/`.
Run `streamlit run app/app.py`; the yellow "Demo mode" banner disappears.

Note: samples are picked as correctly predicted validation images so the demo is clean — say so if asked, and show the
confusion matrix for the honest picture.